# Experiment 54 - Exp22 Rich MLP Features

**Building on:** exp53 (OOF cmAP=0.94973), the best exp22-family plateau config so far.

**Hypothesis:** Later experiments found useful sequence features for MLP probes, but also moved into taxon-specific priors that published poorly. This isolates the feature-engineering part only: keep the exp22/exp53 prior and ensemble style, replace basic MLP temporal features with richer min/rank/quantile/position/diff/activity features.


In [ ]:
import json
from pathlib import Path

nb_dir = Path.cwd() if Path.cwd().name == 'notebooks' else Path.cwd() / 'notebooks'
exp49_path = nb_dir / 'exp49_exp22_public_aware.ipynb'
exp49_nb = json.loads(exp49_path.read_text())

# Reuse exp49 setup through post-processing helpers.
for cell_idx in range(1, 7):
    print(f'--- executing exp49 cell {cell_idx} ---')
    src = ''.join(exp49_nb['cells'][cell_idx]['source'])
    exec(compile(src, f'exp49_cell_{cell_idx}', 'exec'), globals())


In [ ]:
# Rich MLP probes: exp22 model size with exp40-42 style sequence features

def build_seq_features(scores_col, n_windows=N_WINDOWS):
    x = scores_col.reshape(-1, n_windows)
    prev = np.concatenate([x[:, :1], x[:, :-1]], axis=1).reshape(-1)
    next_ = np.concatenate([x[:, 1:], x[:, -1:]], axis=1).reshape(-1)
    mean = np.repeat(x.mean(axis=1), n_windows)
    max_ = np.repeat(x.max(axis=1), n_windows)
    std = np.repeat(x.std(axis=1), n_windows)
    min_ = np.repeat(x.min(axis=1), n_windows)
    p25 = np.repeat(np.percentile(x, 25, axis=1), n_windows)
    p75 = np.repeat(np.percentile(x, 75, axis=1), n_windows)
    score_rng = max_ - min_
    rank = np.zeros_like(x, dtype=np.float32)
    order = np.argsort(np.argsort(x, axis=1), axis=1).astype(np.float32)
    if n_windows > 1:
        rank = order / float(n_windows - 1)
    rank = rank.reshape(-1)
    win_pos = np.tile(np.linspace(0.0, 1.0, n_windows, dtype=np.float32), x.shape[0])
    diff = (next_ - prev).astype(np.float32)
    med = np.repeat(np.median(x, axis=1), n_windows)
    above_med = (scores_col > med).astype(np.float32)
    count_active = np.repeat((x > x.mean(axis=1, keepdims=True)).sum(axis=1), n_windows).astype(np.float32) / n_windows
    return prev, next_, mean, max_, std, min_, rank, p25, p75, win_pos, diff, count_active, above_med, score_rng

def train_mlp_probes(emb, scores_raw, Y, pca_dim=64, alpha_blend=0.25, min_pos=5):
    scaler = StandardScaler()
    emb_s = scaler.fit_transform(emb)
    pca = PCA(n_components=min(pca_dim, emb_s.shape[1] - 1))
    Z = pca.fit_transform(emb_s).astype(np.float32)
    print(f'  PCA: {emb.shape} -> {Z.shape}  ({pca.explained_variance_ratio_.sum():.2%})')
    probe_models = {}
    active = np.where(Y.sum(axis=0) >= min_pos)[0]
    for ci in active:
        y = Y[:, ci]
        if y.min() == y.max():
            continue
        prev, next_, mean, max_, std, min_, rank, p25, p75, win_pos, diff, count_active, above_med, score_rng = build_seq_features(scores_raw[:, ci])
        X = np.hstack([
            Z, scores_raw[:, ci:ci+1], prev[:, None], next_[:, None], mean[:, None], max_[:, None], std[:, None],
            min_[:, None], rank[:, None], p25[:, None], p75[:, None], win_pos[:, None], diff[:, None],
            count_active[:, None], above_med[:, None], score_rng[:, None],
        ])
        pos = max(int(y.sum()), 1)
        neg = len(y) - pos
        sw = np.where(y == 1, min(neg / pos, 25.0), 1.0).astype(np.float32)
        clf = MLPClassifier(hidden_layer_sizes=CFG['mlp_hidden'], activation='relu', solver='adam',
                            alpha=1e-3, learning_rate_init=1e-3, max_iter=150,
                            early_stopping=True, validation_fraction=0.20, n_iter_no_change=12,
                            random_state=42)
        clf.fit(X, y, sample_weight=sw)
        probe_models[ci] = clf
    print(f'  Trained {len(probe_models)} rich probes')
    return probe_models, scaler, pca, alpha_blend

def apply_mlp_probes(emb_test, scores_test, probe_models, scaler, pca, alpha_blend=0.25):
    Z_test = pca.transform(scaler.transform(emb_test)).astype(np.float32)
    result = scores_test.copy()
    for ci, clf in probe_models.items():
        prev, next_, mean, max_, std, min_, rank, p25, p75, win_pos, diff, count_active, above_med, score_rng = build_seq_features(scores_test[:, ci])
        X_test = np.hstack([
            Z_test, scores_test[:, ci:ci+1], prev[:, None], next_[:, None], mean[:, None], max_[:, None], std[:, None],
            min_[:, None], rank[:, None], p25[:, None], p75[:, None], win_pos[:, None], diff[:, None],
            count_active[:, None], above_med[:, None], score_rng[:, None],
        ])
        prob = np.clip(clf.predict_proba(X_test)[:, 1], 1e-5, 1 - 1e-5)
        logit = np.log(prob / (1 - prob)).astype(np.float32)
        result[:, ci] = (1 - alpha_blend) * scores_test[:, ci] + alpha_blend * logit
    return result.astype(np.float32)

print('Rich MLP probes ready')


In [ ]:
# Reuse LightProtoSSM definition and OOF loop from exp49; cell 9 will use the rich MLP functions above.
for cell_idx in [8, 9]:
    print(f'--- executing exp49 cell {cell_idx} ---')
    src = ''.join(exp49_nb['cells'][cell_idx]['source'])
    exec(compile(src, f'exp49_cell_{cell_idx}', 'exec'), globals())


In [ ]:
# Exp54 search around exp53 plateau config

LAMBDAS_EXT = [3.0, 4.0, 5.0]
TTA_W0S = [0.85, 0.90]
POWERS_EXT = [1.0, 1.1, 1.2]
ALPHAS = [0.08, 0.10]
ENSEMBLE_GRIDS = [
    (0.02, 0.58, 0.40),  # exp53 best
    (0.02, 0.63, 0.35),
    (0.05, 0.55, 0.40),
    (0.05, 0.60, 0.35),
    (0.00, 0.60, 0.40),
]

best_cmap = 0.0
best_cfg = {}
results = []
prior_cache = {}

def get_prior_pair_exp54(lam):
    if lam not in prior_cache:
        prior_cache[lam] = (
            apply_prior(oof_raw_0, oof_meta_sites, oof_meta_hours, prior_tables, lam,
                        prior_shrink=1.0, prior_logit_clip=None),
            apply_prior(oof_raw_250, oof_meta_sites, oof_meta_hours, prior_tables, lam,
                        prior_shrink=1.0, prior_logit_clip=None),
        )
    return prior_cache[lam]

for lam in LAMBDAS_EXT:
    prior_0, prior_250 = get_prior_pair_exp54(lam)
    for wp, wm, wpr in ENSEMBLE_GRIDS:
        fp_0 = wp * oof_proto_0 + wm * oof_mlp_0 + wpr * prior_0
        fp_250 = wp * oof_proto_250 + wm * oof_mlp_250 + wpr * prior_250
        for tta_w0 in TTA_W0S:
            fp_avg = tta_w0 * fp_0 + (1.0 - tta_w0) * fp_250
            for power in POWERS_EXT:
                for alpha in ALPHAS:
                    probs = postprocess(fp_avg, power=power, base_alpha=alpha)
                    cmap = padded_cmap(Y_FULL_aligned, probs)
                    row = {'lam': lam, 'tta_w0': tta_w0, 'power': power, 'alpha': alpha,
                           'wp': wp, 'wm': wm, 'wpr': wpr, 'cmap': cmap}
                    results.append(row)
                    if cmap > best_cmap:
                        best_cmap = cmap
                        best_cfg = row.copy()

df = pd.DataFrame(results).sort_values('cmap', ascending=False).reset_index(drop=True)
print(f'Best OOF cmAP: {best_cmap:.5f}')
print(f'Best config:   {best_cfg}')
print('\nTop 20 configs:')
print(df.head(20).to_string(index=False))

lam_b = best_cfg['lam']; tta_w0_b = best_cfg['tta_w0']
pow_b = best_cfg['power']; alp_b = best_cfg['alpha']
wp_b = best_cfg['wp']; wm_b = best_cfg['wm']; wpr_b = best_cfg['wpr']
prior_0_f, prior_250_f = get_prior_pair_exp54(lam_b)
fp_0_f = wp_b * oof_proto_0 + wm_b * oof_mlp_0 + wpr_b * prior_0_f
fp_250_f = wp_b * oof_proto_250 + wm_b * oof_mlp_250 + wpr_b * prior_250_f
p_best = postprocess(tta_w0_b * fp_0_f + (1.0 - tta_w0_b) * fp_250_f,
                     power=pow_b, base_alpha=alp_b)
auc_b = macro_auc(Y_FULL_aligned, p_best)
cmap_b = padded_cmap(Y_FULL_aligned, p_best)

print('=' * 60)
print('Experiment 54 - Exp22 rich MLP features')
print(f'Best: lambda={lam_b}, tta_w0={tta_w0_b}, power={pow_b}, alpha={alp_b}')
print(f'Weights: wp={wp_b}, wm={wm_b}, wpr={wpr_b}')
print(f'AUC={auc_b:.5f}  cmAP={cmap_b:.5f}')
print(f'vs exp53 OOF (0.94973): delta cmAP = {cmap_b - 0.94973:+.5f}')
print(f'vs exp22 OOF (0.93894): delta cmAP = {cmap_b - 0.93894:+.5f}')
print(f'Total wall time: {(time.time() - _WALL_START)/60:.1f} min')
